In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

import os

In [2]:
data_dir = "./data"

for dirname, _, filenames in os.walk(data_dir):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./data/test.csv
./data/train.csv


In [3]:
train_data = pd.read_csv(os.path.join(data_dir, "train.csv"))
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
test_data = pd.read_csv(os.path.join(data_dir, "test.csv"))
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [6]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


In [7]:
y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=500, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print("CV accuracy:", scores.mean(), "±", scores.std(), scores)

submission = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
submission.to_csv('simple_rf.csv', index=False)
submission.head()

CV accuracy: 0.799051186017478 ± 0.029328879930349046 [0.84444444 0.76404494 0.75280899 0.79775281 0.7752809  0.79775281
 0.78651685 0.80898876 0.84269663 0.82022472]


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [8]:
def add_features(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["CabinKnown"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("U")

    df["Surname"] = (
        df["Name"]
        .str.split(",", n=1)
        .str[0]
        .str.strip()
        .str.lower()
    )

    df["IsChild"] = (
        (df["Age"].fillna(99) < 16) |
        (df["Title"].eq("Master"))
    ).astype(int)

    df["IsWomanChild"] = (
        df["Sex"].eq("female") |
        df["IsChild"].eq(1)
    ).astype(int)

    df["WCG_Group"] = np.where(
        df["IsWomanChild"].eq(1),
        df["Surname"] + "_" + df["Pclass"].astype(str),
        "NONE"
    )

    df["Ticket_Group"] = np.where(
        df["IsWomanChild"].eq(1),
        df["Ticket"].astype(str),
        "NONE"
    )
    return df

train_fe = add_features(train_data)
test_fe = add_features(test_data)
train_fe.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Title,FamilySize,IsAlone,CabinKnown,Deck,Surname,IsChild,IsWomanChild,WCG_Group,Ticket_Group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,Mr,2,0,0,U,braund,0,0,NONE,NONE
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,Mrs,2,0,1,C,cumings,0,1,cumings_1,PC 17599
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,Miss,1,1,0,U,heikkinen,0,1,heikkinen_3,STON/O2. 3101282
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,Mrs,2,0,1,C,futrelle,0,1,futrelle_1,113803
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,Mr,1,1,0,U,allen,0,0,NONE,NONE


In [9]:
def add_group_survival_features(train_df, test_df, y, group_cols):
    train_df = train_df.copy()
    test_df = test_df.copy()

    # Align y with train_df before filtering
    train_with_y = train_df.copy()
    train_with_y["Survived"] = y.values

    for col in group_cols:
        stats = (
            train_with_y.loc[train_with_y[col] != "NONE", [col, "Survived"]]
            .groupby(col)["Survived"]
            .agg(["mean", "count"])
        )

        for df in [train_df, test_df]:
            rate = df[col].map(stats["mean"]).fillna(-1)
            count = df[col].map(stats["count"]).fillna(0)

            df[f"{col}_SurvivalRate"] = rate
            df[f"{col}_TrainCount"] = count
            df[f"{col}_Known"] = (rate != -1).astype(int)
            df[f"{col}_AllSurvived"] = ((rate == 1) & (count >= 1)).astype(int)
            df[f"{col}_AllDied"] = ((rate == 0) & (count >= 1)).astype(int)

    return train_df, test_df

train_fe, test_fe = add_group_survival_features(
    train_fe,
    test_fe,
    y,
    group_cols=["WCG_Group", "Ticket_Group"]
)

In [10]:
y = train_fe["Survived"]

features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "Title", "FamilySize", "IsAlone", "CabinKnown", "Deck"
]

# wcg_features = [
#     "IsChild",
#     "IsWomanChild",

#     "WCG_Group_SurvivalRate",
#     "WCG_Group_TrainCount",
#     "WCG_Group_Known",
#     "WCG_Group_AllSurvived",
#     "WCG_Group_AllDied",

#     "Ticket_Group_SurvivalRate",
#     "Ticket_Group_TrainCount",
#     "Ticket_Group_Known",
#     "Ticket_Group_AllSurvived",
#     "Ticket_Group_AllDied",
# ]

# features += wcg_features

X = train_fe[features]
X_test = test_fe[features]

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "CabinKnown", "Pclass"]
categorical_features = ["Sex", "Embarked", "Title", "Deck"]

# numeric_features += wcg_features

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=5,
    random_state=42
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
print("CV accuracy:", scores.mean(), "±", scores.std(), scores)

clf.fit(X, y)
predictions = clf.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions
})

submission.to_csv("good_rf.csv", index=False)
submission.head()

CV accuracy: 0.8327663046889713 ± 0.005662863369867861 [0.83798883 0.8258427  0.8258427  0.83707865 0.83707865]


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [11]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(
        n_estimators=500, max_depth=5, random_state=42
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500, max_depth=5, random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])
    scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")

LogisticRegression: 0.8260 ± 0.0348
RandomForest: 0.8327 ± 0.0229
ExtraTrees: 0.8226 ± 0.0258
GradientBoosting: 0.8338 ± 0.0290
